# Image Representation Selection (Diagnostic 3b formulation, img_proj_dim=4)


In [1]:
import sys
from pathlib import Path

IN_COLAB = 'google.colab' in sys.modules

if IN_COLAB:
    from google.colab import drive
    drive.mount('/content/drive')
    PROJECT_ROOT = Path('/content/drive/MyDrive/dissertation')
else:
    PROJECT_ROOT = Path('../../..').resolve()

sys.path.insert(0, str(PROJECT_ROOT))

print(f"IN_COLAB={IN_COLAB}  REPO_ROOT={PROJECT_ROOT}")

Mounted at /content/drive
IN_COLAB=True  REPO_ROOT=/content/drive/MyDrive/dissertation


In [3]:
import json

import numpy as np

from src.models.classical_baseline import HORIZONS

SEEDS = [7, 14, 21, 28, 35, 42, 49, 56, 63, 70]
IMG_PROJ_DIM = 4

BASE_DIR = PROJECT_ROOT / 'experiments' / '3_multimodal' / 'multimodal' / 'results' / 'gated_cgm_time'
BASE_STUB = 'gated_cgm_time'


ENCODER_SOURCES = {
    'resnet18': (PROJECT_ROOT / 'experiments' / '4_diagnostic' / 'results' / 'diagnostic_3b',
                 'diagnostic3b_resnet18_dim4'),
    'vit':      (PROJECT_ROOT / 'experiments' / '5_residual_multimodal' / 'results' / 'residual_vit',
                 'residual_vit_dim4'),
    'clip':     (PROJECT_ROOT / 'experiments' / '5_residual_multimodal' / 'results' / 'residual_clip',
                 'residual_clip_dim4'),
    'dinov2':   (PROJECT_ROOT / 'experiments' / '5_residual_multimodal' / 'results' / 'residual_dinov2',
                 'residual_dinov2_dim4'),
}


def load_seed_results(result_dir, stub, seeds, result_key='val_results'):
    loaded = {}
    for seed in seeds:
        path = result_dir / f'{stub}_seed{seed}_results.json'
        if not path.exists():
            raise FileNotFoundError(f'Missing result file for seed {seed}: {path}')
        with open(path) as f:
            result = json.load(f)
        if result_key not in result:
            raise KeyError(f'{path.name} missing "{result_key}". Available: {list(result.keys())}')
        loaded[seed] = result
    return loaded


def rmse_by_seed(loaded_results, seeds, horizons, result_key='val_results'):
    return {
        h: np.asarray([float(loaded_results[seed][result_key][str(h)]['rmse']) for seed in seeds],
                       dtype=float)
        for h in horizons
    }


base_loaded = load_seed_results(BASE_DIR, BASE_STUB, SEEDS)
base_by_h = rmse_by_seed(base_loaded, SEEDS, HORIZONS)

encoder_by_h = {}
for encoder, (result_dir, stub) in ENCODER_SOURCES.items():
    loaded = load_seed_results(result_dir, stub, SEEDS)
    encoder_by_h[encoder] = rmse_by_seed(loaded, SEEDS, HORIZONS)

print(f'Loaded CGM+time baseline ({len(SEEDS)} seeds) and '
      f'{len(encoder_by_h)} encoders x {len(SEEDS)} seeds, all at img_proj_dim={IMG_PROJ_DIM}.')


Loaded CGM+time baseline (10 seeds) and 4 encoders x 10 seeds, all at img_proj_dim=4.


In [4]:
def paired_stats(base_by_h, candidate_by_h, horizons, seeds):
    """Per-horizon mean/sd RMSE, paired improvement % vs. CGM+time, and win count (out of
    len(seeds)) -- "win" means this encoder's residual model beat CGM+time for that seed."""
    out = {}
    for h in horizons:
        base = base_by_h[h]
        cand = candidate_by_h[h]
        improvement_pct = (base - cand) / base * 100.0
        wins = int(np.sum(cand < base))
        out[h] = {
            'candidate_rmse_mean': float(np.mean(cand)),
            'candidate_rmse_sd': float(np.std(cand, ddof=1)),
            'improvement_pct_mean': float(np.mean(improvement_pct)),
            'improvement_pct_sd': float(np.std(improvement_pct, ddof=1)),
            'wins': wins,
            'n_seeds': len(seeds),
        }
    return out


per_encoder_stats = {
    encoder: paired_stats(base_by_h, encoder_by_h[encoder], HORIZONS, SEEDS)
    for encoder in encoder_by_h
}

print('=== Per-horizon paired improvement over CGM+time (validation, img_proj_dim=4) ===')
print(f"{'encoder':>10}{'horizon':>10}{'improvement %':>20}{'wins/seeds':>14}")
for encoder in ENCODER_SOURCES:
    for h in HORIZONS:
        s = per_encoder_stats[encoder][h]
        print(f"{encoder:>10}{h:>8} min"
              f"{s['improvement_pct_mean']:>+11.3f} +/- {s['improvement_pct_sd']:<6.3f}"
              f"{s['wins']:>8}/{s['n_seeds']}")
    print()


=== Per-horizon paired improvement over CGM+time (validation, img_proj_dim=4) ===
   encoder   horizon       improvement %    wins/seeds
  resnet18      15 min     +0.112 +/- 0.078       10/10
  resnet18      30 min     +0.208 +/- 0.056       10/10
  resnet18      60 min     +0.424 +/- 0.091       10/10
  resnet18      90 min     +0.488 +/- 0.126       10/10
  resnet18     120 min     +0.622 +/- 0.143       10/10

       vit      15 min     +0.060 +/- 0.042        9/10
       vit      30 min     +0.074 +/- 0.071        8/10
       vit      60 min     +0.078 +/- 0.149        8/10
       vit      90 min     +0.086 +/- 0.211        8/10
       vit     120 min     +0.184 +/- 0.190        8/10

      clip      15 min     +0.203 +/- 0.089       10/10
      clip      30 min     +0.388 +/- 0.070       10/10
      clip      60 min     +0.639 +/- 0.141       10/10
      clip      90 min     +0.700 +/- 0.179       10/10
      clip     120 min     +0.821 +/- 0.161       10/10

    dinov2      15 m

In [5]:
print('=== Selection: mean improvement % across 5 horizons + overall win-rate ===')
print(f"{'encoder':>10}{'mean improvement %':>22}{'overall win-rate':>20}")

summary_rows = []
for encoder in ENCODER_SOURCES:
    stats = per_encoder_stats[encoder]
    mean_improvement = float(np.mean([stats[h]['improvement_pct_mean'] for h in HORIZONS]))
    total_wins = sum(stats[h]['wins'] for h in HORIZONS)
    total_pairs = sum(stats[h]['n_seeds'] for h in HORIZONS)
    win_rate = total_wins / total_pairs
    summary_rows.append({
        'encoder': encoder,
        'mean_improvement_pct': mean_improvement,
        'total_wins': total_wins,
        'total_pairs': total_pairs,
        'win_rate': win_rate,
    })
    print(f"{encoder:>10}{mean_improvement:>+20.3f}%{total_wins:>13}/{total_pairs} ({win_rate:.1%})")

summary_rows.sort(key=lambda r: r['mean_improvement_pct'], reverse=True)
best = summary_rows[0]

print(f"\nRanked by mean improvement %: {[r['encoder'] for r in summary_rows]}")
print(f"Selected encoder: {best['encoder']} "
      f"(mean improvement={best['mean_improvement_pct']:+.3f}%, "
      f"win-rate={best['win_rate']:.1%})")

agreement = all(r['win_rate'] == 1.0 for r in summary_rows if r is not summary_rows[-1]) or True
n_full_consistency = sum(1 for r in summary_rows if r['win_rate'] == 1.0)
print(f"\n{n_full_consistency}/{len(summary_rows)} encoders improve CGM+time in all "
      f"{best['total_pairs']} horizon x seed pairs (100% win-rate); ranking among those is "
      f"therefore decided purely by effect size (mean improvement %).")


=== Selection: mean improvement % across 5 horizons + overall win-rate ===
   encoder    mean improvement %    overall win-rate
  resnet18              +0.371%           50/50 (100.0%)
       vit              +0.096%           41/50 (82.0%)
      clip              +0.550%           50/50 (100.0%)
    dinov2              +0.630%           50/50 (100.0%)

Ranked by mean improvement %: ['dinov2', 'clip', 'resnet18', 'vit']
Selected encoder: dinov2 (mean improvement=+0.630%, win-rate=100.0%)

3/4 encoders improve CGM+time in all 50 horizon x seed pairs (100% win-rate); ranking among those is therefore decided purely by effect size (mean improvement %).


In [7]:
RESULTS_DIR = PROJECT_ROOT / 'experiments' / '6_best_image_encoder' / 'results'
RESULTS_DIR.mkdir(parents=True, exist_ok=True)
OUTPUT_PATH = RESULTS_DIR / 'encoder_selection_summary.json'

with open(OUTPUT_PATH, 'w') as f:
    json.dump({
        'seeds': SEEDS,
        'img_proj_dim': IMG_PROJ_DIM,
        'per_horizon': {
            encoder: {str(h): per_encoder_stats[encoder][h] for h in HORIZONS}
            for encoder in ENCODER_SOURCES
        },
        'overall_ranking': summary_rows,
        'selected_encoder': best['encoder'],
    }, f, indent=2)

print(f'Saved encoder selection summary to: {OUTPUT_PATH}')


Saved encoder selection summary to: /content/drive/MyDrive/dissertation/experiments/6_best_image_encoder/results/encoder_selection_summary.json
